In [1]:
import os
from pathlib import Path

HOME_DIR = os.path.dirname(os.getcwd())
DATA_DIR = Path("/data/dacon-236105")
OUTPUT_DIR = Path(HOME_DIR) / "output"
LOG_DIR = Path(HOME_DIR) / "logs"
ENV_PATH = Path(HOME_DIR) / ".env"

OUTPUT_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# EDA 7. Zero-shot SER with OpenAI `gpt-audio-1.5` (prompt only, no training)

- Chat Completions에 wav를 `input_audio`(base64)로 넣고, 6개 감정 중 하나를 한 단어로 답하게 함
- API key는 `.env`의 `OPENAI_API_KEY`에서 읽음 (`.env`는 gitignore 대상)
- 파일 1개 응답 → `logs/temp_openai_{train,test}.csv`에 1행씩 즉시 append → 재실행 시 이미 기록된 id는 건너뜀
- train 전체로 accuracy / macro-F1 측정, test 예측은 `output/openai_gpt_audio.csv`

## Import

In [2]:
import base64
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from tqdm.auto import tqdm

/home/hommy/Repositories/dacon-236105/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Hyperparameter Setting

In [3]:
CFG = {
    "MODEL": "gpt-audio-1.5",
    "MODALITIES": ["text"],  # 텍스트 응답만 요청 (audio 출력 없음)
    "TEMPERATURE": 0.0,      # 모델 페이지에 지원 여부 미기재 → 400 에러가 나면 None으로 (요청에서 제외)
    "N_WORKERS": 4,          # 동시 요청 수
    "MAX_RETRIES": 5,        # SDK 자동 재시도 (429 / 5xx / 연결 오류, exponential backoff)
    "TIMEOUT": 60,           # 요청당 timeout (초)
    "FALLBACK_LABEL": 4,     # 응답을 파싱하지 못한 test 파일의 제출 label (neutral)

    # USD per 1M tokens (gpt-audio-1.5 모델 페이지 기준)
    "PRICE_TEXT_IN": 2.5,
    "PRICE_AUDIO_IN": 32.0,
    "PRICE_TEXT_OUT": 10.0,
}

LABELS = ["angry", "fear", "sad", "disgust", "neutral", "happy"]  # label index 0..5 (프롬프트 보기 + confusion matrix 표시용)

## Data

In [4]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

train_df["path"] = train_df["path"].apply(lambda p: str(DATA_DIR / p))
test_df["path"] = test_df["path"].apply(lambda p: str(DATA_DIR / p))

labels = train_df["label"].values
len(train_df), len(test_df)

(5001, 1881)

## OpenAI Client / Prompt

In [5]:
load_dotenv(ENV_PATH)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"
client = OpenAI(max_retries=CFG["MAX_RETRIES"], timeout=CFG["TIMEOUT"])

OPTIONS = '\n'.join(f"{i}: {name}" for i, name in enumerate(LABELS))
PROMPT = (
    "Listen to the speech clip and classify the speaker's emotion "
    "into exactly one of these six categories:\n"
    + OPTIONS
    + "\nJudge from how it is said (tone, pitch, loudness, speaking rate, voice quality).\n"
    "Answer with only the category number (0-5), with no other text."
)
print(PROMPT)

def parse_label(text):
    """응답에 등장하는 숫자가 0..5 중 정확히 하나면 그 값, 아니면 -1 (예: "3", "3." → 3 / "2 or 4", "7" → -1)."""
    numbers = set(re.findall(r"\d+", text))
    return int(numbers.pop()) if len(numbers) == 1 and numbers <= set("012345") else -1

def classify(file_id, path):
    with open(path, "rb") as f:
        audio_b64 = base64.b64encode(f.read()).decode("utf-8")
    extra = {} if CFG["TEMPERATURE"] is None else {"temperature": CFG["TEMPERATURE"]}
    response = client.chat.completions.create(
        model=CFG["MODEL"],
        modalities=CFG["MODALITIES"],
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT},
                {"type": "input_audio", "input_audio": {"data": audio_b64, "format": "wav"}},
            ],
        }],
        **extra,
    )
    text = response.choices[0].message.content or ''
    usage = response.usage
    details = getattr(usage, "prompt_tokens_details", None)
    return {
        "id": file_id,
        "pred": parse_label(text),
        "response": text.strip(),
        "prompt_tokens": usage.prompt_tokens,
        "audio_tokens": getattr(details, "audio_tokens", None) or 0,
        "completion_tokens": usage.completion_tokens,
        "time": datetime.now().isoformat(timespec="seconds"),
    }

Listen to the speech clip and classify the speaker's emotion into exactly one of these six categories:
0: angry
1: fear
2: sad
3: disgust
4: neutral
5: happy
Judge from how it is said (tone, pitch, loudness, speaking rate, voice quality).
Answer with only the category number (0-5), with no other text.


## Resumable Inference

In [6]:
LOG_COLUMNS = ["id", "pred", "response", "prompt_tokens", "audio_tokens", "completion_tokens", "time"]

def read_log(log_path):
    if not log_path.exists():
        return pd.DataFrame(columns=LOG_COLUMNS)
    return pd.read_csv(log_path, keep_default_na=False).drop_duplicates("id", keep="last")

def append_log(log_path, row):
    pd.DataFrame([row], columns=LOG_COLUMNS).to_csv(
        log_path, mode='a', header=not log_path.exists(), index=False,
    )

LOG_LOCK = threading.Lock()

def classify_and_log(file_id, path, log_path):
    row = classify(file_id, path)
    with LOG_LOCK:  # 응답을 받은 worker가 바로 1행 append
        append_log(log_path, row)

def run_inference(df, log_path):
    """log에 이미 있는 id는 건너뛰고, 응답이 올 때마다 log에 1행 append (끊겨도 재실행하면 이어서 진행)."""
    done = set(read_log(log_path)["id"])
    todo = df[~df["id"].isin(done)]
    print(f"{log_path.name}: {len(done)} done, {len(todo)} to request")
    executor = ThreadPoolExecutor(max_workers=CFG["N_WORKERS"])
    futures = [executor.submit(classify_and_log, i, p, log_path) for i, p in zip(todo["id"], todo["path"])]
    try:
        for future in tqdm(as_completed(futures), total=len(futures)):
            future.result()  # 재시도 소진 / 인증 오류 등은 예외로 중단 (대기 중 요청은 취소, 이미 받은 응답은 기록됨)
    finally:
        executor.shutdown(wait=True, cancel_futures=True)
    return df[["id"]].merge(read_log(log_path), on="id", how="left")

def api_cost(log):
    audio_in = log["audio_tokens"].sum()
    text_in = log["prompt_tokens"].sum() - audio_in
    text_out = log["completion_tokens"].sum()
    return (text_in * CFG["PRICE_TEXT_IN"] + audio_in * CFG["PRICE_AUDIO_IN"]
            + text_out * CFG["PRICE_TEXT_OUT"]) / 1e6

## Train Evaluation (전체 5001개)

In [7]:
train_log = run_inference(train_df, LOG_DIR / "temp_openai_train.csv")
assert train_log["pred"].notna().all()

train_pred = train_log["pred"].astype(int).values
n_unparsed = (train_pred == -1).sum()
acc = accuracy_score(labels, train_pred)
macro_f1 = f1_score(labels, train_pred, labels=list(range(len(LABELS))), average="macro")
cost = api_cost(train_log)
print(f"[train] accuracy {acc:.4f} | macro-F1 {macro_f1:.4f} | unparsed {n_unparsed}/{len(train_pred)} | cost ${cost:.2f}")

cm = confusion_matrix(labels, train_pred, labels=[-1] + list(range(len(LABELS))))[1:]
display(pd.DataFrame(cm, index=LABELS, columns=["unparsed"] + LABELS))

temp_openai_train.csv: 0 done, 5001 to request


100%|██████████| 5001/5001 [17:21<00:00,  4.80it/s]

[train] accuracy 0.1922 | macro-F1 0.1286 | unparsed 0/5001 | cost $5.19


,unparsed,angry,fear,sad,disgust,neutral,happy
angry,0,65,1,3,0,788,10
fear,0,3,32,51,0,733,29
sad,0,0,4,97,0,758,0
disgust,0,5,3,42,1,796,5
neutral,0,0,0,7,0,715,0
happy,0,4,7,3,0,788,51


## Test Inference / Submission

In [ ]:
# test_log = run_inference(test_df, LOG_DIR / "temp_openai_test.csv")
# assert test_log["pred"].notna().all()

# test_pred = test_log["pred"].astype(int).values
# n_unparsed = (test_pred == -1).sum()
# fallback = CFG["FALLBACK_LABEL"]
# test_pred[test_pred == -1] = fallback
# cost = api_cost(test_log)
# print(f"[test] unparsed {n_unparsed}/{len(test_pred)} → label {fallback} | cost ${cost:.2f}")

# submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
# submission["label"] = submission["id"].map(pd.Series(test_pred, index=test_log["id"].values))
# assert submission["label"].notna().all()
# submission["label"] = submission["label"].astype(int)
# submission.to_csv(OUTPUT_DIR / "openai_gpt_audio.csv", index=False)
# submission["label"].value_counts().sort_index()